In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import os
import pandas as pd 
import numpy as np
import typing as tp
import re
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
%matplotlib inline
import os
import seaborn as sns; sns.set_style("white")
import umap as umap

# Set a random seed
import random
rng = np.random.RandomState(123)

# Set current working directory


In [ ]:
# Parameters. run_all.py sweeps these via the environment so one notebook can
# emit every combination; the defaults keep interactive use unchanged.
import os
# Set up the parameters
# For plotting
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
dpi = 300
figformat = 'pdf'

# Create a color map for the pathways
pathways = pd.Series(['MAPK','Cell Cycle', 'DNA Damage', 
                      'PI3K/Akt/mTOR', 'Epigenetics', 'Stem Cells & Wnt', 
                      'Angiogenesis', 'Protein Tyrosine Kinase', 
                      'Apoptosis', 'JAK/STAT', 'Cytoskeletal Signaling', 
                      'TGF-beta/Smad', 'Others', 'Proteases'])
colors = sns.color_palette('tab20', len(pathways))
lut = dict(zip(pathways, colors))

# For the data
cell_line = os.environ.get('COLOPAINT3D_CELL_LINE', 'HCT116')
data_type = os.environ.get('COLOPAINT3D_DATA_TYPE', 'aggregates')  # 'aggregates', 'MIP' or '2D'
name = f'{cell_line} {data_type}'

grit_threshold = 1.96

# Paper panel for this (cell_line, data_type, embedding) combination. The notebook is
# run once per combination; save_panel routes each to the right figure folder.
PANEL = {
    ("HCT116", "MIP",        "supervised"):   "Fig4a",
    ("HCT116", "aggregates", "supervised"):   "Fig4b",
    ("HCT116", "MIP",        "unsupervised"): "Fig4c",
    ("HCT116", "aggregates", "unsupervised"): "Fig4d",
    ("HT29",   "MIP",        "supervised"):   "SupplFig4a",
    ("HT29",   "aggregates", "supervised"):   "SupplFig4b",
    ("HT29",   "MIP",        "unsupervised"): "SupplFig4c",
    ("HT29",   "aggregates", "unsupervised"): "SupplFig4d",
    # NOTE: no 2D outputs survive upstream, so these two are UNVERIFIED.
    ("HCT116", "2D",         "supervised"):   "Fig5b",
    ("HT29",   "2D",         "supervised"):   "SupplFig5a",
}

random_state = 42
print(f'cell_line={cell_line}  data_type={data_type}')

# Not every (cell_line, data_type, embedding) combination is a paper panel: the 2D
# runs have a supervised panel (Fig 5b / Suppl 5a) but no unsupervised counterpart.
# Look up rather than index, so a non-panel combination skips instead of raising
# after the panels earlier in the notebook have already been written.
def panel_for(cell_line, data_type, embedding):
    name = PANEL.get((cell_line, data_type, embedding))
    if name is None:
        print(f'no paper panel for ({cell_line}, {data_type}, {embedding}) — not saving')
    return name


In [ ]:
# Helper functions
def is_meta_column(
    c:str,
    allowlist:tp.List[str]=["Metadata_Well","Metadata_Barcode","Metadata_AcqID","Metadata_Site"],
    denylist:tp.List[str]=[],
)->bool:
    """
        allowlist:
            the function will return False for these, no matter if they are metadata or not
        denylist:
            the function will return True for these, no matter if they are metadata or not
    """
    if c in allowlist:
        return False
    if c in denylist:
        return True
    for ex in '''
        Metadata
        ^Count
        ImageNumber
        Object
        Parent
        Children
        Plate
        Well
        Location
        _[XYZ]_
        _[XYZ]$
        BoundingBox
        Phase
        Orientation
        Angle
        Scale
        Scaling
        Width
        Height
        Group
        FileName
        PathName
        URL
        Execution
        ModuleError
        LargeBrightArtefact
        MD5Digest
        RadialDistribution_Frac
        Intensity_
        _Manders
        _Overflow
    '''.split():
        if re.search(ex, c):
            return True
    return False

def get_feature_matrix(df):
    feature_cols = [c for c in df.columns if not is_meta_column(c)]
    return df[feature_cols].select_dtypes(include=[np.number]).values

def makePCA(df, n_components=2, lut=None):
    df = df.copy()
    dataN = get_feature_matrix(df)
    pca_model = PCA(n_components=n_components)
    pca_model = pca_model.fit(dataN)
    pcaOut = pca_model.transform(dataN)

    for i in range(n_components):
        df[f'pc{i+1}'] = pcaOut[:, i]
    
    return df

def makeUMAP(df, is_supervised=False, n_neighbors=200, min_dist=0.1, spread=5, 
                metric='cosine', random_state=random_state, use_pca=False, n_components=2):
    df = df.copy()
    dataN = get_feature_matrix(df)

    if use_pca:
        pca_model = PCA(n_components=n_components)
        dataN = pca_model.fit_transform(dataN)

    umap_model = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, spread=spread,
                            metric=metric, random_state=random_state, n_jobs=-1)

    if is_supervised:
        umapOut = umap_model.fit_transform(dataN, y=df['Metadata_cmpd_onehot'].values)
    else:
        umapOut = umap_model.fit_transform(dataN)

    df['umap1'] = umapOut[:, 0]
    df['umap2'] = umapOut[:, 1]

    return df

def plot_embedding(df, x_col, y_col, name='', lut=None, pathways=None, 
                     grit_threshold=grit_threshold, alpha_bg=0.2, alpha_fg=0.8):

      fig, ax = plt.subplots(figsize=(3,2), dpi=dpi)
      ax.set_xlabel(x_col.upper().replace('PC', 'PC ').replace('UMAP', 'UMAP '), fontsize=10)
      ax.set_ylabel(y_col.upper().replace('PC', 'PC ').replace('UMAP', 'UMAP '), fontsize=10)
      ax.spines[['top', 'right']].set_color('w')
      ax.spines[['left', 'bottom']].set_color('grey')
      ax.set_facecolor('w')

      # If grit filtering: background + foreground layers
      if grit_threshold:
          sns.scatterplot(data=df, x=x_col, y=y_col, hue='Metadata_pathway',
                         palette=lut, marker='.', alpha=alpha_bg, linewidth=0, ax=ax)
          plot_df = df[df['Metadata_grit'] >= grit_threshold]
          sns.scatterplot(data=plot_df, x=x_col, y=y_col, hue='Metadata_pathway',
                         palette=lut, marker='.', alpha=alpha_fg, linewidth=0, legend=False, ax=ax)
      else:
          # No filtering: just plot all data
          plot_df = df
          sns.scatterplot(data=plot_df, x=x_col, y=y_col, hue='Metadata_pathway',
                         palette=lut, marker='.', linewidth=0, alpha=alpha_fg, ax=ax)

      if pathways is not None: 
      # Numbered centroids
      
        pathways = sorted(plot_df['Metadata_pathway'].unique())
        pathway_to_num = {p: i for i, p in enumerate(pathways, start=1)}
        for pathway, num in pathway_to_num.items():
            center = plot_df[plot_df['Metadata_pathway'] == pathway][[x_col, y_col]].mean()
            ax.text(center[x_col], center[y_col], str(num), fontsize=8,
                    ha='right', va='bottom', bbox=dict(facecolor='none', edgecolor='none'))

      # Numbered legend
        ax.get_legend().remove()
        handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=lut[p], markersize=8)
                for p in pathways if p in plot_df['Metadata_pathway'].values]
        labels = [f"{pathway_to_num[p]:<3} {p}" for p in pathways if p in plot_df['Metadata_pathway'].values]
        ax.legend(handles, labels, loc='best', bbox_to_anchor=(1, 1))

      if pathways is None:
          ax.get_legend().remove()

      ax.set_title(name)
      return fig, ax

In [ ]:
# Load the data
dir = str(profiles("exp1_main", "")) + "/"
data =  pd.read_parquet(('{}grit_data_{}_{}.parquet').format(dir, data_type, cell_line))

dataset = data.copy()
dataset = dataset[dataset['Metadata_pert_type'] == 'trt']
dataset = dataset.dropna(axis='columns', how='all')

# Implement one-hot encoding for the pathways
onehot_mapping = {name: i for i, name in enumerate(dataset['Metadata_pathway'].unique())}
df = dataset.copy()

df['Metadata_cmpd_onehot'] = df['Metadata_pathway'].map(onehot_mapping)

In [ ]:
dataset[['Metadata_pathway', 'Metadata_cmpdname']].drop_duplicates().sort_values('Metadata_pathway')



#### Start plotting


In [ ]:
## PCA
results_pca = makePCA(df, n_components=20, lut=lut)

fig, ax = plot_embedding(results_pca, x_col='pc1', y_col='pc2', name=name, lut=lut, 
                         pathways=pathways, grit_threshold=grit_threshold,
                         )

# Change the marker size for better visibility
for collection in ax.collections:
    collection.set_sizes([15])

ax.legend_.remove()


In [ ]:
## Supervised UMAP

results_supervised = makeUMAP(df,
                                is_supervised=True,
                                n_neighbors=200,
                                use_pca=False,
                                random_state=random_state)

fig, ax = plot_embedding(results_supervised,
                         x_col='umap1',
                         y_col='umap2',
                         name=f'{cell_line} - {data_type}',
                         lut=lut,
                         pathways=pathways,
                         grit_threshold=grit_threshold,
)

ax.set_xticklabels([])
ax.set_yticklabels([])
# ax.legend_.remove()

# Change the marker size for better visibility
for collection in ax.collections:
    collection.set_sizes([5])


_panel = panel_for(cell_line, data_type, "supervised")
if _panel:
    save_panel(fig, _panel,
               data=results_supervised,
               caption=f"Supervised UMAP coloured by pathway, {cell_line} {data_type}",
               notebook="analysis/3_Figure4/PCAUMAP_pathway_v2.ipynb")

In [ ]:
# Filter data based on grit threshold
df = df[df['Metadata_grit'] >= grit_threshold]

In [ ]:
## UMAP without supervision

results_unsupervised = makeUMAP(df,
                                is_supervised=False,
                                n_neighbors=200,
                                use_pca=False,
                                random_state=random_state) 
fig, ax = plot_embedding(results_unsupervised,
                         x_col='umap1',
                         y_col='umap2',
                         name=name,
                         lut=lut,
                         pathways=pathways,
                         grit_threshold=grit_threshold,
                         )  

ax.set_xticklabels([])
ax.set_yticklabels([])

# Change the marker size for better visibility
for collection in ax.collections:
    collection.set_sizes([15]) 

ax.legend().remove()

# fig.savefig(
#         "3_Figure4/PCAUMAP/result-images/UMAP_unsupervised_pathway_{}_{}.{}".format(cell_line, data_type,figformat), dpi=dpi, bbox_inches="tight"
#         )

#### Investigate a subset

In [ ]:
# Focused visualization on a specific pathway in supervised UMAP

zoomin = "Cell Cycle" 
df_subset = results_unsupervised[results_unsupervised['Metadata_pathway'].str.contains(zoomin)]


fig, ax = plt.subplots(figsize=(2,4))

sns.scatterplot(data=df_subset, x='umap1', y='umap2', 
                hue='Metadata_cmpdname', ax=ax, alpha=0.8, marker='.')

ax.set_xlabel('UMAP 1', fontsize=10)
ax.set_ylabel('UMAP 2', fontsize=10)
ax.spines['top'].set_color('w')
ax.spines['right'].set_color('w')
ax.spines['left'].set_color('grey')
ax.spines['bottom'].set_color('grey')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))

# Change the marker size for better visibility
for collection in ax.collections:
    collection.set_sizes([100]) 

fig.show()

# fig.savefig(
#         "3_Figure4/PCAUMAP/result-images/UMAP_pathway_{}_{}_{}.{}".format(cell_line, data_type,zoomin,figformat), dpi=dpi, bbox_inches="tight"
#         )

plt.show()
plt.close()

#### Quantification now on umap data

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Prepare data for quantitative evaluation
X = get_feature_matrix(df)
pathway_labels = df['Metadata_pathway'].to_numpy()
le = LabelEncoder()
pathway_numeric = le.fit_transform(pathway_labels)
n_pathways = len(np.unique(pathway_numeric))


In [ ]:
## Filter out pathways that have fewer than 10 occurrences

# Count pathway occurrences
pathway_counts = pd.Series(pathway_labels).value_counts()
print("Pathway distribution:")
print(pathway_counts)

# Filter out pathways with low occurrence
min_compounds = 10  
keep_mask = np.array([count >= min_compounds for pathway in pathway_labels 
                      for count in [pathway_counts[pathway]]])

# Filter the data
X_filtered = X[keep_mask]
pathway_labels_filtered = pathway_labels[keep_mask]
pathway_numeric_filtered = le.fit_transform(pathway_labels_filtered)
n_pathways_filtered = len(np.unique(pathway_labels_filtered))


print(f"\nOriginal: {len(pathway_labels)} compounds, {n_pathways} pathways")
print(f"Filtered: {len(pathway_labels_filtered)} compounds, {n_pathways_filtered} pathways")
print(f"Removed pathways: {set(pathway_labels) - set(pathway_labels_filtered)}")

In [ ]:
## Calculate metrics on filtered embedding

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

umap_model = umap.UMAP(n_components=2, random_state=random_state, metric='cosine', n_neighbors=200, spread=5, min_dist=0.1)
embedding = umap_model.fit_transform(X_filtered)

km = KMeans(n_clusters=n_pathways_filtered, random_state=random_state, n_init=10)
pred_labels = km.fit_predict(embedding)

silhouette = silhouette_score(embedding, pred_labels)
ARI = adjusted_rand_score(pathway_numeric_filtered, pred_labels)
NMI = normalized_mutual_info_score(pathway_numeric_filtered, pred_labels)

print("\n=== Metrics ===")
print(f"  Metrics - ARI: {ARI:.3f}, NMI: {NMI:.3f}, Silhouette: {silhouette:.3f}")


In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Clusters from best UMAP configuration shown on 2D projection
ax = axes[0]
scatter = ax.scatter(embedding[:, 0], embedding[:, 1], 
                    c=pred_labels, cmap='tab10', alpha=0.7, s=30)
ax.set_title('KMeans Clusters on UMAP Embedding')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
plt.colorbar(scatter, ax=ax, label='Cluster')

# Plot 2: True pathway labels
ax = axes[1]
sns.scatterplot(x=embedding[:, 0], y=embedding[:, 1], 
                    hue=pathway_labels_filtered, palette=lut, alpha=0.7, s=50, ax=ax)
ax.set_title('True Pathways')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')

ax.set_xticklabels([])
ax.set_yticklabels([])


ax.legend(title="Pathways", bbox_to_anchor=(1.05, 1), loc='upper left')

_panel = panel_for(cell_line, data_type, "unsupervised")
if _panel:
    save_panel(fig, _panel,
               data=pd.DataFrame({"umap1": embedding[:, 0], "umap2": embedding[:, 1],
                                  "kmeans_cluster": pred_labels,
                                  "pathway": pathway_labels_filtered}),
               caption=f"Unsupervised UMAP, KMeans clusters vs pathway, {cell_line} {data_type}",
               notebook="analysis/3_Figure4/PCAUMAP_pathway_v2.ipynb")

In [ ]:
# Per-pathway analysis with NMI
print("\n=== Per-Pathway Analysis (Binary) ===")
pathway_metrics = []
for pathway in np.unique(pathway_labels_filtered):
    # Binary: this pathway vs all others
    binary_true = (pathway_labels_filtered == pathway).astype(int)

    # Find clusters enriched for this pathway - use best_k from grid search
    binary_pred = np.zeros_like(binary_true)
    for cluster_id in range(n_pathways_filtered):
        cluster_mask = pred_labels == cluster_id
        pathway_enrichment = np.mean(pathway_labels_filtered[cluster_mask] == pathway)
        overall_pathway_freq = np.mean(pathway_labels_filtered == pathway)

        if pathway_enrichment > 1.5 * overall_pathway_freq:
            binary_pred[cluster_mask] = 1

    pathway_metrics.append({
        'Pathway': pathway,
        'Count': np.sum(pathway_labels_filtered == pathway),
        'ARI': adjusted_rand_score(binary_true, binary_pred),
        'NMI': normalized_mutual_info_score(binary_true, binary_pred)
    })

pathway_df = pd.DataFrame(pathway_metrics).sort_values('NMI', ascending=False)
print(pathway_df.to_string())

# Save the per-pathway metrics
pathway_df.to_csv(__import__("os").makedirs(str(ROOT / 'analysis' / '3_Figure4' / 'results'), exist_ok=True) or str(ROOT / 'analysis' / '3_Figure4' / 'results') + "/per_pathway_metrics_{}_{}.csv".format(cell_line, data_type), index=False)

In [ ]:
from tqdm import tqdm


def permutation_test(X, true_labels, n_permutations=1000, n_clusters=7, n_components=2, random_state=42):
    """
    Perform permutation test for clustering significance
    """
    np.random.seed(random_state)
    
    # Get actual clustering metrics
    umap_model = umap.UMAP(n_components=n_components, random_state=random_state, n_neighbors=200, spread=5, min_dist=0.1, metric='cosine')
    embedding = umap_model.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=20)
    clusters = kmeans.fit_predict(embedding)
    
    actual_ari = adjusted_rand_score(true_labels, clusters)
    actual_nmi = normalized_mutual_info_score(true_labels, clusters)
    
    # Permutation tests
    perm_aris = []
    perm_nmis = []
    
    print(f"Running {n_permutations} permutations...")
    for i in tqdm(range(n_permutations)):
        # Shuffle labels
        shuffled_labels = true_labels.copy()
        np.random.shuffle(shuffled_labels)
        
        # Calculate metrics with shuffled labels
        perm_ari = adjusted_rand_score(shuffled_labels, clusters)
        perm_nmi = normalized_mutual_info_score(shuffled_labels, clusters)
        
        perm_aris.append(perm_ari)
        perm_nmis.append(perm_nmi)
    
    # Calculate p-values
    p_value_ari = np.mean(np.array(perm_aris) >= actual_ari)
    p_value_nmi = np.mean(np.array(perm_nmis) >= actual_nmi)
    
    return {
        'actual_ari': actual_ari,
        'actual_nmi': actual_nmi,
        'perm_aris': perm_aris,
        'perm_nmis': perm_nmis,
        'p_value_ari': p_value_ari,
        'p_value_nmi': p_value_nmi
    }

# Run permutation test on filtered data
results = permutation_test(X_filtered, pathway_numeric_filtered, 
                          n_permutations=1000, n_clusters=n_pathways_filtered, n_components=2)

print("\n=== Permutation Test Results ===")
print(f"Actual ARI: {results['actual_ari']:.3f}")
print(f"Permuted ARI mean: {np.mean(results['perm_aris']):.3f} ± {np.std(results['perm_aris']):.3f}")
print(f"P-value (ARI): {results['p_value_ari']:.4f}")
print(f"\nActual NMI: {results['actual_nmi']:.3f}")
print(f"Permuted NMI mean: {np.mean(results['perm_nmis']):.3f} ± {np.std(results['perm_nmis']):.3f}")
print(f"P-value (NMI): {results['p_value_nmi']:.4f}")

# Effect size (z-score)
z_score_ari = (results['actual_ari'] - np.mean(results['perm_aris'])) / np.std(results['perm_aris'])
z_score_nmi = (results['actual_nmi'] - np.mean(results['perm_nmis'])) / np.std(results['perm_nmis'])
print(f"\nEffect size (z-score):")
print(f"ARI: {z_score_ari:.2f}")
print(f"NMI: {z_score_nmi:.2f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot ARI distribution
ax = axes[0]
ax.hist(results['perm_aris'], bins=50, alpha=0.7, edgecolor='black', label='Permuted')
ax.axvline(results['actual_ari'], color='red', linewidth=2, label=f'Actual ({results["actual_ari"]:.3f})')
ax.axvline(np.mean(results['perm_aris']), color='blue', linestyle='--', 
          label=f'Perm mean ({np.mean(results["perm_aris"]):.3f})')
ax.set_xlabel('ARI Score')
ax.set_ylabel('Frequency')
ax.set_title(f'Permutation Test: ARI\np-value = {results["p_value_ari"]:.4f}')
ax.legend()

# Plot NMI distribution
ax = axes[1]
ax.hist(results['perm_nmis'], bins=50, alpha=0.7, edgecolor='black', label='Permuted')
ax.axvline(results['actual_nmi'], color='red', linewidth=2, label=f'Actual ({results["actual_nmi"]:.3f})')
ax.axvline(np.mean(results['perm_nmis']), color='blue', linestyle='--',
          label=f'Perm mean ({np.mean(results["perm_nmis"]):.3f})')
ax.set_xlabel('NMI Score')
ax.set_ylabel('Frequency')
ax.set_title(f'Permutation Test: NMI\np-value = {results["p_value_nmi"]:.4f}')
ax.legend()

plt.tight_layout()
plt.show()

# Save the permutation test numbers
with open(str(ROOT / 'analysis' / '3_Figure4' / 'results') + "/permutation_test_results_{}_{}.txt".format(cell_line, data_type), 'w') as f:
    f.write("=== Permutation Test Results ===\n")
    f.write(f"Actual ARI: {results['actual_ari']:.6f}\n")
    f.write(f"Permuted ARI mean: {np.mean(results['perm_aris']):.6f} ± {np.std(results['perm_aris']):.6f}\n")
    f.write(f"P-value (ARI): {results['p_value_ari']:.6f}\n\n")
    f.write(f"Actual NMI: {results['actual_nmi']:.6f}\n")
    f.write(f"Permuted NMI mean: {np.mean(results['perm_nmis']):.6f} ± {np.std(results['perm_nmis']):.6f}\n")
    f.write(f"P-value (NMI): {results['p_value_nmi']:.6f}\n\n")
    f.write(f"Effect size (z-score):\n")
    f.write(f"ARI: {z_score_ari:.2f}\n")
    f.write(f"NMI: {z_score_nmi:.2f}\n")
